# Qwen3 QLoRA Fine-Tuning on MLX (Apple Silicon)

This notebook fine-tunes **Qwen3 on MLX** for insurance claim classification.

It uses the claim-level SFT dataset from the `gemma4` pipeline, converted to MLX chat format.

## Dataset Format

The dataset uses MLX-LM `chat` format:
```json
{"messages": [{"role": "system", "content": "..."}, {"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
```

## Hardware Settings

Optimized for **48 GB M4 Pro**:
- `batch_size=1`
- `grad_accumulation_steps=4`
- `num_layers=8`
- `max_seq_length=2048`
- `grad_checkpoint=True`


## 1) Install dependencies

In [ ]:
%pip install -U "mlx-lm[train]" datasets huggingface_hub pyyaml jupyterlab python-dotenv google-generativeai scikit-learn

In [ ]:
from pathlib import Path
import importlib.metadata as ilm
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
from pprint import pprint

# Project paths - pointing to the qwen3 directory
PROJECT_ROOT = Path("/Users/fzuin/nlp-dataset")
QWEN3_DIR = PROJECT_ROOT / "qwen3"
DATA_DIR = QWEN3_DIR / "data"
EXPORTS_DIR = DATA_DIR / "exports"

# Training output paths
ADAPTER_PATH = QWEN3_DIR / "adapters" / "qwen3_claim_sft"
FUSED_MODEL_DIR = QWEN3_DIR / "fused_model" / "qwen3_claim_sft"
CONFIG_PATH = QWEN3_DIR / "qwen3_claim_sft.yaml"

# Model configuration
MODEL_NAME = "mlx-community/Qwen3-8B-4bit"
FALLBACK_MODEL_NAME = "mlx-community/Qwen3-4B-Instruct-2507-4bit"

# Conservative training settings for 48 GB unified memory
NUM_LAYERS = 8
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 4
ITERS = 600
LEARNING_RATE = 1e-5
MAX_SEQ_LENGTH = 2048
VAL_BATCHES = 25
TEST_BATCHES = 100
SAVE_EVERY = 100
STEPS_PER_REPORT = 10
STEPS_PER_EVAL = 100
GRAD_CHECKPOINT = True
MASK_PROMPT = True

# LoRA parameters
LORA_RANK = 16
LORA_SCALE = 32.0
LORA_DROPOUT = 0.05
LORA_KEYS = ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj"]

for path in [ADAPTER_PATH.parent, FUSED_MODEL_DIR.parent]:
    path.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("QWEN3_DIR:", QWEN3_DIR)
print("DATA_DIR:", DATA_DIR)
print("EXPORTS_DIR:", EXPORTS_DIR)
print("ADAPTER_PATH:", ADAPTER_PATH)

print("\nInstalled versions:")
for pkg in ["mlx-lm", "datasets", "huggingface_hub"]:
    try:
        print(f"  {pkg}: {ilm.version(pkg)}")
    except Exception:
        print(f"  {pkg}: not found")

## 2) Check and prepare dataset

In [ ]:
# Check if the exported chat format data exists
train_chat_path = EXPORTS_DIR / "claim_sft_train_chat.jsonl"
val_chat_path = EXPORTS_DIR / "claim_sft_val_chat.jsonl"
test_chat_path = EXPORTS_DIR / "claim_sft_test_chat.jsonl"

print("Checking exported chat format data...")
print(f"  train_chat: {train_chat_path.exists()} - {train_chat_path}")
print(f"  val_chat: {val_chat_path.exists()} - {val_chat_path}")
print(f"  test_chat: {test_chat_path.exists()} - {test_chat_path}")

if not train_chat_path.exists():
    print("\n⚠️ Chat format data not found. Running export script...")
    # Run the export script
    import sys
    sys.path.insert(0, str(PROJECT_ROOT))
    from qwen3.export_chat_format import convert_dataset, save_json
    from qwen3.common import DEFAULT_TRAIN_PATH, DEFAULT_VAL_PATH, DEFAULT_TEST_PATH, DEFAULT_CHAT_EXPORT_DIR
    
    # Convert the data
    train_stats = convert_dataset(DEFAULT_TRAIN_PATH, train_chat_path)
    val_stats = convert_dataset(DEFAULT_VAL_PATH, val_chat_path)
    test_stats = convert_dataset(DEFAULT_TEST_PATH, test_chat_path)
    
    print(f"Converted train: {train_stats['converted']} records")
    print(f"Converted val: {val_stats['converted']} records")
    print(f"Converted test: {test_stats['converted']} records")
else:
    # Count records
    def count_lines(path):
        with open(path, 'r') as f:
            return sum(1 for _ in f)
    
    print(f"\nDataset sizes:")
    print(f"  Train: {count_lines(train_chat_path)} samples")
    print(f"  Val: {count_lines(val_chat_path)} samples")
    print(f"  Test: {count_lines(test_chat_path)} samples")

In [ ]:
# Validate a sample from the dataset
import json

def validate_chat_record(record):
    """Validate that a chat record has the correct format."""
    if "messages" not in record:
        return False, "Missing 'messages' field"
    
    messages = record["messages"]
    if not isinstance(messages, list) or len(messages) < 2:
        return False, "'messages' must have at least 2 entries"
    
    roles = [m.get("role") for m in messages]
    if "user" not in roles:
        return False, "Missing 'user' message"
    if "assistant" not in roles:
        return False, "Missing 'assistant' message"
    
    return True, "Valid"

# Check samples
with open(train_chat_path, 'r') as f:
    sample = json.loads(f.readline())
    
valid, msg = validate_chat_record(sample)
print(f"Sample validation: {msg}")
print(f"\nSample record structure:")
print(json.dumps(sample, indent=2, ensure_ascii=False)[:1500] + "...")

## 3) Write MLX-LM training config

In [ ]:
import yaml

# Use the chat-format data directory
DATA_DIR_FOR_TRAINING = str(EXPORTS_DIR)

config = {
    "model": MODEL_NAME,
    "train": True,
    "fine_tune_type": "lora",
    "optimizer": "adamw",
    "data": DATA_DIR_FOR_TRAINING,
    "seed": 42,
    "num_layers": NUM_LAYERS,
    "batch_size": BATCH_SIZE,
    "iters": ITERS,
    "val_batches": VAL_BATCHES,
    "learning_rate": LEARNING_RATE,
    "steps_per_report": STEPS_PER_REPORT,
    "steps_per_eval": STEPS_PER_EVAL,
    "grad_accumulation_steps": GRAD_ACCUMULATION_STEPS,
    "adapter_path": str(ADAPTER_PATH),
    "save_every": SAVE_EVERY,
    "test": False,
    "test_batches": TEST_BATCHES,
    "max_seq_length": MAX_SEQ_LENGTH,
    "grad_checkpoint": GRAD_CHECKPOINT,
    "mask_prompt": MASK_PROMPT,
    "lora_parameters": {
        "keys": LORA_KEYS,
        "rank": LORA_RANK,
        "scale": LORA_SCALE,
        "dropout": LORA_DROPOUT,
    },
}

with CONFIG_PATH.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False, allow_unicode=True)

print(f"Config saved to: {CONFIG_PATH}")
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 4) Helper functions

In [ ]:
def run_cmd(cmd, cwd=None, env=None):
    """Run a command and stream output."""
    env_full = os.environ.copy()
    if env:
        env_full.update(env)

    print("Running command:")
    print(" ".join(map(str, cmd)))
    print()

    process = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env_full,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in process.stdout:
            print(line, end="")
    finally:
        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")

    return return_code

## 5) (Optional) Test base model generation

In [ ]:
# Test with a claim from the dataset
TEST_PROMPT = """You are an insurance claim consistency analyst. You receive one structured accident claim with detected damages and party statements. Return valid JSON only. Estimate the probability that the claim is true, choose verdict=true or verdict=not_true, explain the decision briefly, and list the main incongruences if they exist.

Claim:
Claim ID: PT-001
Location: Porto
Incident type: rear-end collision
Detected damages: bumper, rear light

Statements:
1. Role: insured
   Vehicle: car A
   Text: I was stopped at a red light when the other car hit me from behind.

2. Role: third_party
   Vehicle: car B
   Text: The car in front braked suddenly and I couldn't stop in time.

JSON response:"""

run_cmd([
    sys.executable, "-m", "mlx_lm", "generate",
    "--model", MODEL_NAME,
    "--prompt", TEST_PROMPT,
    "--max-tokens", "300",
])

## 6) Start QLoRA training

In [ ]:
# This will train Qwen3 on the claim classification dataset
print(f"Training with data from: {DATA_DIR_FOR_TRAINING}")
print(f"Adapter will be saved to: {ADAPTER_PATH}")
print(f"\nStarting training...")

run_cmd([
    sys.executable, "-m", "mlx_lm", "lora",
    "--config", str(CONFIG_PATH),
])

## 7) Evaluate on test set

In [ ]:
if test_chat_path.exists():
    run_cmd([
        sys.executable, "-m", "mlx_lm", "lora",
        "--model", MODEL_NAME,
        "--adapter-path", str(ADAPTER_PATH),
        "--data", str(EXPORTS_DIR),
        "--test",
        "--test-batches", str(TEST_BATCHES),
        "--batch-size", str(BATCH_SIZE),
        "--max-seq-length", str(MAX_SEQ_LENGTH),
    ])
else:
    print("No test.jsonl found. Skipping evaluation.")

## 8) Generate with fine-tuned model

In [ ]:
# Test the fine-tuned model on the same claim
print("Testing fine-tuned model on claim...")

run_cmd([
    sys.executable, "-m", "mlx_lm", "generate",
    "--model", MODEL_NAME,
    "--adapter-path", str(ADAPTER_PATH),
    "--prompt", TEST_PROMPT,
    "--max-tokens", "300",
])

## 9) (Optional) Fuse adapters into base model

In [ ]:
run_cmd([
    sys.executable, "-m", "mlx_lm", "fuse",
    "--model", MODEL_NAME,
    "--adapter-path", str(ADAPTER_PATH),
    "--save-path", str(FUSED_MODEL_DIR),
])

## Notes

### Memory tuning for 48 GB M4 Pro

If you hit memory issues:
- Switch to `FALLBACK_MODEL_NAME` (smaller model)
- Reduce `NUM_LAYERS` from 8 to 4
- Reduce `MAX_SEQ_LENGTH` from 2048 to 1024
- Keep `BATCH_SIZE = 1`
- Increase `GRAD_ACCUMULATION_STEPS` instead of batch size
- Keep `GRAD_CHECKPOINT = True`

### For better quality

- Increase `NUM_LAYERS` to 12 or 16
- Try `LORA_RANK = 32`
- Increase `ITERS` to 1000+
- Monitor validation loss carefully

### Dataset pipeline

The data comes from the `gemma4` pipeline:
1. `build_claim_sft_source.py` - Process raw claims
2. `generate_claim_sft_teacher.py` - Generate targets with Gemini
3. `split_claim_sft_dataset.py` - Train/val/test split
4. `export_chat_format.py` - Convert to MLX chat format

Re-run these scripts if you need to update the dataset.
